<a href="https://colab.research.google.com/github/icosahedron31/Walmart-Sales/blob/main/model_experiment_XGBoost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mlflow dagshub xgboost -q

import pandas as pd
import numpy as np
import xgboost as xgb
import mlflow
import mlflow.sklearn
import dagshub
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import LabelEncoder
import itertools

dagshub.init(repo_owner='icosahedron31', repo_name='Walmart-Sales', mlflow=True)

mlflow.set_experiment('XGBoost_Training')

Accessing as njvar23

Initialized MLflow to track repo "icosahedron31/Walmart-Sales"

Repository icosahedron31/Walmart-Sales initialized!

<Experiment: artifact_location='mlflow-artifacts:/ccb8f3212aa349f78ebd9351b7e0fcc3', creation_time=1783411223370, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1783411223370, lifecycle_stage='active', name='XGBoost_Training', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

train = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/train_processed.csv')
val = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/val_processed.csv')

train['Date'] = pd.to_datetime(train['Date'])
val['Date'] = pd.to_datetime(val['Date'])

print(train.shape)
print(val.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(332778, 29)
(88792, 29)


In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/walmart/Transformers')
from LabelEncoder import LabelEncoderTransformer

In [ ]:
train = train.dropna(subset=['lag_52', 'rolling_mean_52'])
val = val.dropna(subset=['lag_52', 'rolling_mean_52'])

print(train.shape)
print(val.shape)

(173973, 29)
(87110, 29)


In [ ]:
print(train.columns.tolist())

['Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment', 'Type', 'Size', 'Week', 'Month', 'Year', 'Quarter', 'holiday_name', 'lag_1', 'lag_2', 'lag_4', 'lag_52', 'rolling_mean_4', 'rolling_mean_12', 'rolling_std_4', 'rolling_mean_52']


In [ ]:
feature_cols = [
    'Store', 'Dept', 'Week', 'Month', 'Year', 'Quarter',
    'IsHoliday', 'holiday_name', 'Type', 'Size',
    'Temperature', 'Fuel_Price', 'CPI', 'Unemployment',
    'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5',
    'lag_52', 'rolling_mean_52', 'lag_1', 'lag_2', 'lag_4',
    'rolling_mean_4', 'rolling_mean_12', 'rolling_std_4'
]

train = train.dropna(subset=['lag_52', 'rolling_mean_52'])
val = val.dropna(subset=['lag_52', 'rolling_mean_52'])
print(f"Train: {train.shape}, Val: {val.shape}")

X_train = train[feature_cols].copy()
y_train = train['Weekly_Sales']
is_holiday_train = train['IsHoliday']

X_val = val[feature_cols].copy()
y_val = val['Weekly_Sales']
is_holiday_val = val['IsHoliday']

X_train['IsHoliday'] = X_train['IsHoliday'].astype(int)
X_val['IsHoliday'] = X_val['IsHoliday'].astype(int)

# Sample weights matching WMAE's holiday weighting, for training too
sample_weight_train = np.where(is_holiday_train, 5, 1)

print(f"Features: {len(feature_cols)}, Train: {X_train.shape}, Val: {X_val.shape}")

Train: (173973, 29), Val: (87110, 29)
Features: 27, Train: (173973, 27), Val: (87110, 27)


In [ ]:
X_train = X_train.copy()
X_val = X_val.copy()

# IsHoliday is boolean — convert to int
X_train['IsHoliday'] = X_train['IsHoliday'].astype(int)
X_val['IsHoliday'] = X_val['IsHoliday'].astype(int)

In [ ]:
def wmae(y_true, y_pred, is_holiday):
    weights = is_holiday.map({True: 5, False: 1})
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

In [ ]:
with mlflow.start_run(run_name='XGBoost_Cleaning'):
    mlflow.log_param('train_rows', len(X_train))
    mlflow.log_param('val_rows', len(X_val))
    mlflow.log_param('num_features', len(feature_cols))
    mlflow.log_param('cutoff_date', '2012-04-01')
    mlflow.log_param('categorical_encoding', 'LabelEncoder')
    print("Cleaning run logged")

Cleaning run logged
🏃 View run XGBoost_Cleaning at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/2/runs/5ea0eb6d63af477d9e54e69306025b56
🧪 View experiment at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/2


In [ ]:
with mlflow.start_run(run_name='XGBoost_Feature_Selection'):

    baseline_pipeline = Pipeline([
        ('encoder', LabelEncoderTransformer(['Type', 'holiday_name'])),
        ('model', xgb.XGBRegressor(
            n_estimators=100,
            learning_rate=0.1,
            random_state=42,
            verbosity=0
        ))
    ])
    baseline_pipeline.fit(X_train, y_train)

    importance_df = pd.DataFrame({
        'feature': feature_cols,
        'importance': baseline_pipeline.named_steps['model'].feature_importances_
    }).sort_values('importance', ascending=False)
    print(importance_df)

    val_preds = baseline_pipeline.predict(X_val)
    is_holiday_mask = is_holiday_val.values.astype(bool)

    score_wmae = wmae(y_val, val_preds, is_holiday_val)
    score_mae = np.mean(np.abs(y_val.values - val_preds))
    score_mae_holiday = np.mean(np.abs(y_val.values[is_holiday_mask] - val_preds[is_holiday_mask]))
    score_mae_non_holiday = np.mean(np.abs(y_val.values[~is_holiday_mask] - val_preds[~is_holiday_mask]))
    score_rmse = np.sqrt(np.mean((y_val.values - val_preds) ** 2))

    mlflow.log_param('top_5_features', importance_df['feature'].head(5).tolist())
    mlflow.log_param('num_features', len(feature_cols))
    mlflow.log_metric('wmae', score_wmae)
    mlflow.log_metric('mae', score_mae)
    mlflow.log_metric('mae_holiday', score_mae_holiday)
    mlflow.log_metric('mae_non_holiday', score_mae_non_holiday)
    mlflow.log_metric('rmse', score_rmse)

    print(f"\nBaseline WMAE: {score_wmae:.2f} | MAE: {score_mae:.2f} | "
          f"Holiday MAE: {score_mae_holiday:.2f} | Non-holiday MAE: {score_mae_non_holiday:.2f} | RMSE: {score_rmse:.2f}")

            feature  importance
19           lag_52    0.475814
21            lag_1    0.323023
24   rolling_mean_4    0.078661
7      holiday_name    0.042362
8              Type    0.015074
23            lag_4    0.008563
25  rolling_mean_12    0.007295
16        MarkDown3    0.006851
22            lag_2    0.004726
20  rolling_mean_52    0.003894
1              Dept    0.003845
13     Unemployment    0.003366
15        MarkDown2    0.003173
18        MarkDown5    0.002855
2              Week    0.002718
17        MarkDown4    0.002676
6         IsHoliday    0.002367
26    rolling_std_4    0.002362
14        MarkDown1    0.002125
11       Fuel_Price    0.001555
0             Store    0.001554
9              Size    0.001428
12              CPI    0.001329
3             Month    0.001224
10      Temperature    0.001073
4              Year    0.000088
5           Quarter    0.000000

Baseline WMAE: 1397.80 | MAE: 1390.20 | Holiday MAE: 1454.71 | Non-holiday MAE: 1387.97 | RMSE: 3239.56

In [ ]:
with mlflow.start_run(run_name='XGBoost_GridSearch'):

    param_grid = {
        "learning_rate": [0.05, 0.1],
        "max_depth": [6, 8, 10],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0]
    }

    encoder = LabelEncoderTransformer(['Type', 'holiday_name'])
    X_train_enc = encoder.fit_transform(X_train)
    X_val_enc = encoder.transform(X_val)

    is_holiday_mask = is_holiday_val.values.astype(bool)

    best_score = float('inf')
    best_params = {}
    best_metrics = {}

    for lr, depth, sub, col in itertools.product(
        param_grid["learning_rate"],
        param_grid["max_depth"],
        param_grid["subsample"],
        param_grid["colsample_bytree"]
    ):
        m = xgb.XGBRegressor(
            n_estimators=1000,
            learning_rate=lr,
            max_depth=depth,
            subsample=sub,
            colsample_bytree=col,
            objective='reg:absoluteerror',
            random_state=42,
            verbosity=0,
            early_stopping_rounds=50
        )
        m.fit(X_train_enc, y_train, eval_set=[(X_val_enc, y_val)], verbose=False)

        preds = m.predict(X_val_enc)

        score_wmae = wmae(y_val, preds, is_holiday_val)
        score_mae = np.mean(np.abs(y_val.values - preds))
        score_mae_holiday = np.mean(np.abs(y_val.values[is_holiday_mask] - preds[is_holiday_mask]))
        score_mae_non_holiday = np.mean(np.abs(y_val.values[~is_holiday_mask] - preds[~is_holiday_mask]))
        score_rmse = np.sqrt(np.mean((y_val.values - preds) ** 2))

        if score_wmae < best_score:
            best_score = score_wmae
            best_params = {'learning_rate': lr, 'max_depth': depth, 'subsample': sub, 'colsample_bytree': col}
            best_metrics = {
                'wmae': score_wmae,
                'mae': score_mae,
                'mae_holiday': score_mae_holiday,
                'mae_non_holiday': score_mae_non_holiday,
                'rmse': score_rmse,
                'best_iteration': m.best_iteration
            }

        print(f"lr={lr}, depth={depth}, sub={sub}, col={col} → "
              f"WMAE: {score_wmae:.2f} | MAE: {score_mae:.2f} | "
              f"Holiday MAE: {score_mae_holiday:.2f} | Non-holiday MAE: {score_mae_non_holiday:.2f}")

    mlflow.log_params(best_params)
    mlflow.log_metrics(best_metrics)
    print(f"\nBest params: {best_params}")
    print(f"Best metrics: {best_metrics}")

lr=0.05, depth=6, sub=0.8, col=0.8 → WMAE: 1341.74 | MAE: 1337.09 | Holiday MAE: 1376.56 | Non-holiday MAE: 1335.72
lr=0.05, depth=6, sub=0.8, col=1.0 → WMAE: 1353.59 | MAE: 1347.88 | Holiday MAE: 1396.37 | Non-holiday MAE: 1346.21
lr=0.05, depth=6, sub=1.0, col=0.8 → WMAE: 1360.71 | MAE: 1352.32 | Holiday MAE: 1423.51 | Non-holiday MAE: 1349.86
lr=0.05, depth=6, sub=1.0, col=1.0 → WMAE: 1398.63 | MAE: 1388.49 | Holiday MAE: 1474.50 | Non-holiday MAE: 1385.52
lr=0.05, depth=8, sub=0.8, col=0.8 → WMAE: 1344.25 | MAE: 1335.99 | Holiday MAE: 1406.13 | Non-holiday MAE: 1333.57
lr=0.05, depth=8, sub=0.8, col=1.0 → WMAE: 1368.44 | MAE: 1358.64 | Holiday MAE: 1441.83 | Non-holiday MAE: 1355.77
lr=0.05, depth=8, sub=1.0, col=0.8 → WMAE: 1348.40 | MAE: 1340.30 | Holiday MAE: 1409.03 | Non-holiday MAE: 1337.93
lr=0.05, depth=8, sub=1.0, col=1.0 → WMAE: 1354.35 | MAE: 1346.12 | Holiday MAE: 1415.96 | Non-holiday MAE: 1343.71
lr=0.05, depth=10, sub=0.8, col=0.8 → WMAE: 1342.16 | MAE: 1333.73 | Hol

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

train_sorted = train.sort_values('Date').reset_index(drop=True)
X_train_cv_raw = train_sorted[feature_cols].copy()
y_train_cv = train_sorted['Weekly_Sales']
is_holiday_cv = train_sorted['IsHoliday']

# Fit ONE encoder on the full category domain (Type, holiday_name are fixed, known sets —
# not leakage-sensitive the way numeric features would be)
encoder = LabelEncoderTransformer(['Type', 'holiday_name'])
encoder.fit(X_train_cv_raw)

tscv = TimeSeriesSplit(n_splits=5)
fold_wmae, fold_mae, fold_mae_holiday, fold_mae_non_holiday, fold_rmse = [], [], [], [], []

with mlflow.start_run(run_name='XGBoost_TimeSeriesCV'):
    for fold, (tr_idx, va_idx) in enumerate(tscv.split(X_train_cv_raw)):
        X_tr_raw, X_va_raw = X_train_cv_raw.iloc[tr_idx], X_train_cv_raw.iloc[va_idx]
        y_tr, y_va = y_train_cv.iloc[tr_idx], y_train_cv.iloc[va_idx]
        holiday_va = is_holiday_cv.iloc[va_idx]
        holiday_va_mask = holiday_va.values.astype(bool)

        X_tr = encoder.transform(X_tr_raw)
        X_va = encoder.transform(X_va_raw)

        m = xgb.XGBRegressor(
            n_estimators=1000,
            learning_rate=0.05,
            max_depth=10,
            subsample=0.8,
            colsample_bytree=1.0,
            objective='reg:absoluteerror',
            random_state=42,
            verbosity=0,
            early_stopping_rounds=50
        )
        m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

        preds = m.predict(X_va)

        score_wmae = wmae(y_va, preds, holiday_va)
        score_mae = np.mean(np.abs(y_va.values - preds))
        score_mae_holiday = np.mean(np.abs(y_va.values[holiday_va_mask] - preds[holiday_va_mask])) if holiday_va_mask.sum() > 0 else np.nan
        score_mae_non_holiday = np.mean(np.abs(y_va.values[~holiday_va_mask] - preds[~holiday_va_mask]))
        score_rmse = np.sqrt(np.mean((y_va.values - preds) ** 2))

        fold_wmae.append(score_wmae)
        fold_mae.append(score_mae)
        fold_mae_holiday.append(score_mae_holiday)
        fold_mae_non_holiday.append(score_mae_non_holiday)
        fold_rmse.append(score_rmse)

        print(f"Fold {fold}: train_size={len(X_tr)}, val_size={len(X_va)}, "
              f"holiday_weeks={holiday_va_mask.sum()} → "
              f"WMAE: {score_wmae:.2f} | MAE: {score_mae:.2f} | "
              f"Holiday MAE: {score_mae_holiday:.2f} | RMSE: {score_rmse:.2f}")

    mlflow.log_metric('cv_mean_wmae', np.nanmean(fold_wmae))
    mlflow.log_metric('cv_std_wmae', np.nanstd(fold_wmae))
    mlflow.log_metric('cv_mean_mae', np.nanmean(fold_mae))
    mlflow.log_metric('cv_mean_mae_holiday', np.nanmean(fold_mae_holiday))
    mlflow.log_metric('cv_mean_mae_non_holiday', np.nanmean(fold_mae_non_holiday))
    mlflow.log_metric('cv_mean_rmse', np.nanmean(fold_rmse))

    for i in range(len(fold_wmae)):
        mlflow.log_metric(f'fold_{i}_wmae', fold_wmae[i])
        mlflow.log_metric(f'fold_{i}_mae_holiday', fold_mae_holiday[i])

    print(f"\nCV mean WMAE: {np.nanmean(fold_wmae):.2f} (± {np.nanstd(fold_wmae):.2f})")
    print(f"CV mean MAE: {np.nanmean(fold_mae):.2f}")
    print(f"CV mean Holiday MAE: {np.nanmean(fold_mae_holiday):.2f}")
    print(f"CV mean Non-holiday MAE: {np.nanmean(fold_mae_non_holiday):.2f}")
    print(f"CV mean RMSE: {np.nanmean(fold_rmse):.2f}")

Fold 0: train_size=28998, val_size=28995, holiday_weeks=0 → WMAE: 1586.03 | MAE: 1586.03 | Holiday MAE: nan | RMSE: 3717.06
Fold 1: train_size=57993, val_size=28995, holiday_weeks=0 → WMAE: 1345.62 | MAE: 1345.62 | Holiday MAE: nan | RMSE: 2896.02
Fold 2: train_size=86988, val_size=28995, holiday_weeks=2858 → WMAE: 1610.79 | MAE: 1602.13 | Holiday MAE: 1632.74 | RMSE: 3757.59
Fold 3: train_size=115983, val_size=28995, holiday_weeks=5796 → WMAE: 3457.16 | MAE: 2684.92 | Holiday MAE: 4422.97 | RMSE: 10214.69
Fold 4: train_size=144978, val_size=28995, holiday_weeks=2898 → WMAE: 1436.45 | MAE: 1356.45 | Holiday MAE: 1636.54 | RMSE: 2842.49

CV mean WMAE: 1887.21 (± 791.00)
CV mean MAE: 1715.03
CV mean Holiday MAE: 2564.08
CV mean Non-holiday MAE: 1621.29
CV mean RMSE: 4685.57
🏃 View run XGBoost_TimeSeriesCV at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/2/runs/58b2860e9f0640d3aeb8929ffdc51cb6
🧪 View experiment at: https://dagshub.com/icosahedron31/Walmart-Sales.ml

In [ ]:
with mlflow.start_run(run_name='XGBoost_Pipeline_Final'):

    xgb_pipeline = Pipeline([
        ('encoder', LabelEncoderTransformer(['Type', 'holiday_name'])),
        ('model', xgb.XGBRegressor(
            n_estimators=559,  # from best_iteration found during grid search
            learning_rate=0.05,
            max_depth=10,
            subsample=0.8,
            colsample_bytree=1.0,
            objective='reg:absoluteerror',
            random_state=42,
            verbosity=0
        ))
    ])

    xgb_pipeline.fit(X_train, y_train)

    val_preds_final = xgb_pipeline.predict(X_val)
    is_holiday_mask = is_holiday_val.values.astype(bool)

    score_wmae = wmae(y_val, val_preds_final, is_holiday_val)
    score_mae = np.mean(np.abs(y_val.values - val_preds_final))
    score_mae_holiday = np.mean(np.abs(y_val.values[is_holiday_mask] - val_preds_final[is_holiday_mask]))
    score_mae_non_holiday = np.mean(np.abs(y_val.values[~is_holiday_mask] - val_preds_final[~is_holiday_mask]))
    score_rmse = np.sqrt(np.mean((y_val.values - val_preds_final) ** 2))

    mlflow.log_params({
        'learning_rate': 0.05,
        'max_depth': 10,
        'subsample': 0.8,
        'colsample_bytree': 1.0,
        'objective': 'reg:absoluteerror',
        'n_estimators': 559
    })
    mlflow.log_metric('wmae', score_wmae)
    mlflow.log_metric('mae', score_mae)
    mlflow.log_metric('mae_holiday', score_mae_holiday)
    mlflow.log_metric('mae_non_holiday', score_mae_non_holiday)
    mlflow.log_metric('rmse', score_rmse)

    mlflow.sklearn.log_model(
        xgb_pipeline,
        'xgb_pipeline',
        registered_model_name='XGBoost_Best',
        skops_trusted_types=[
            'LabelEncoder.LabelEncoderTransformer',
            'collections.OrderedDict',
            'xgboost.sklearn.XGBRegressor',
            'xgboost.core.Booster',
            'sklearn.preprocessing._label.LabelEncoder'
        ]
    )

    print(f"Pipeline WMAE: {score_wmae:.2f} | MAE: {score_mae:.2f} | "
          f"Holiday MAE: {score_mae_holiday:.2f} | Non-holiday MAE: {score_mae_non_holiday:.2f} | RMSE: {score_rmse:.2f}")
    print("Pipeline saved and registered as new version of XGBoost_Best")

2026/07/11 17:15:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Registered model 'XGBoost_Best' already exists. Creating a new version of this model...
2026/07/11 17:15:31 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBoost_Best, version 3
Created version '3' of model 'XGBoost_Best'.


Pipeline WMAE: 1341.16 | MAE: 1334.78 | Holiday MAE: 1388.90 | Non-holiday MAE: 1332.91 | RMSE: 2991.22
Pipeline saved and registered as new version of XGBoost_Best
🏃 View run XGBoost_Pipeline_Final at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/2/runs/e30a90ce33aa4e3f836fffc263e9da25
🧪 View experiment at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/2
